In [1]:
%pip install psycopg[binary]

Note: you may need to restart the kernel to use updated packages.


In [2]:
import psycopg

conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="rag_learning",
    user="postgres",
    password="postgres"
)

print("Connected!")

Connected!


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

e:\Coding\playground\rag-from-scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2632.61it/s]


In [4]:
chunks = [
    {
        "id": "chunk_1",
        "text": "Students need an identification card to enter the university library.",
        "metadata": {"page": 10, "section": "Library"}
    },
    {
        "id": "chunk_2",
        "text": "The library is open from 8 AM to 10 PM.",
        "metadata": {"page": 10, "section": "Library"}
    },
    {
        "id": "chunk_3",
        "text": "Students can borrow books from the university library.",
        "metadata": {"page": 11, "section": "Library"}
    }
]

In [8]:
texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(texts)

print(embeddings.shape)

(3, 384)


In [14]:
conn.rollback()

with conn.cursor() as cur:
    cur.execute("SELECT id, text FROM chunks;")
    print(cur.fetchall())

[('chunk_1', 'Students need an identification card to enter the university library.'), ('chunk_2', 'The library is open from 8 AM to 10 PM.'), ('chunk_3', 'Students can borrow books from the university library.')]


In [ ]:
# Insert the chunks + embeddings. to be run only once, otherwise it will create duplicates or fail due to primary key constraint
import json

with conn.cursor() as cur:
    for chunk, embedding in zip(chunks, embeddings):
        cur.execute(
            """
            INSERT INTO chunks (id, text, embedding, metadata)
            VALUES (%s, %s, %s::vector, %s::jsonb);
            """,
            (
                chunk["id"],
                chunk["text"],
                embedding.tolist(),
                json.dumps(chunk["metadata"])
            )
        )

conn.commit()

UniqueViolation: duplicate key value violates unique constraint "chunks_pkey"
DETAIL:  Key (id)=(chunk_1) already exists.

In [19]:
# Verify
with conn.cursor() as cur:
    cur.execute("""
        SELECT id, text, metadata,
               vector_dims(embedding) AS dimensions
        FROM chunks;
    """)
    
    rows = cur.fetchall()

for row in rows:
    print(row)

# DB done

('chunk_1', 'Students need an identification card to enter the university library.', {'page': 10, 'section': 'Library'}, 384)
('chunk_2', 'The library is open from 8 AM to 10 PM.', {'page': 10, 'section': 'Library'}, 384)
('chunk_3', 'Students can borrow books from the university library.', {'page': 11, 'section': 'Library'}, 384)


In [ ]:
conn.rollback() 
# use when db goes into a bad state, e.g. after a failed insert, to reset the connection to a clean state

In [17]:
# Now we embed the query and search for the most similar chunks
query = "When can I use the library?"

query_embedding = model.encode(query)

In [18]:
# Now we search for the most similar chunks using pgvector's cosine distance
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            id,
            text,
            embedding <=> %s::vector AS distance
        FROM chunks
        ORDER BY embedding <=> %s::vector
        LIMIT 2;
        """,
        (query_embedding.tolist(), query_embedding.tolist())
    )

    results = cur.fetchall()

for result in results:
    print(result)

# <=> is the operator for cosine distance in pgvector. The lower the distance, the more similar the vectors are.

('chunk_2', 'The library is open from 8 AM to 10 PM.', 0.39924633502960205)
('chunk_3', 'Students can borrow books from the university library.', 0.5542688039381016)


Indexing techinques

|                | HNSW                | IVFFlat                             |
| -------------- | ------------------- | ----------------------------------- |
| Structure      | Graph               | Clusters/lists                      |
| Search         | Navigate graph      | Search selected clusters            |
| Recall         | Generally excellent | Depends more on tuning              |
| Build          | More expensive      | Generally cheaper                   |
| Memory         | Higher              | Lower                               |
| Updates        | More convenient     | Has some operational considerations |
| Typical choice | ⭐ Great default     | Useful alternative                  |


In [ ]:
# make the index for faster search
with conn.cursor() as cur:
    cur.execute("""
        CREATE INDEX chunks_embedding_hnsw
        ON chunks
        USING hnsw (embedding vector_cosine_ops);
    """)

conn.commit()

In [22]:
# inspect the index
with conn.cursor() as cur:
    cur.execute("""
        SELECT indexname, indexdef
        FROM pg_indexes
        WHERE tablename = 'chunks';
    """)
    print(*cur.fetchall(), sep="\n")

('chunks_pkey', 'CREATE UNIQUE INDEX chunks_pkey ON public.chunks USING btree (id)')
('chunks_embedding_hnsw', 'CREATE INDEX chunks_embedding_hnsw ON public.chunks USING hnsw (embedding vector_cosine_ops)')


In [ ]:
# testing to see if the index is being used
with conn.cursor() as cur:
    cur.execute("""
        EXPLAIN
        SELECT id, text
        FROM chunks
        ORDER BY embedding <=> %s::vector
        LIMIT 2;
    """, (query_embedding.tolist(),))

    for row in cur.fetchall():
        print(row[0])

# it'll be used if the table had more than 1000 rows, but since we only have 3 rows, it won't be used. You can test this by inserting more rows and running the EXPLAIN again.

Limit  (cost=2.06..2.07 rows=2 width=72)
  ->  Sort  (cost=2.06..2.07 rows=3 width=72)
        Sort Key: ((embedding <=> '[-0.0359227,-0.07561592,-0.10080385,-0.00036149955,0.049259078,0.050752565,-0.06455078,0.046266716,0.025749773,-0.038657706,-0.038030874,0.12480196,-0.046150018,0.0015685455,0.042461384,-0.022415534,0.027064899,-0.008798122,0.015232704,-0.034539722,0.04762229,-0.051299363,-0.012794305,-0.027516082,0.011502309,-0.032525115,-0.064711906,-0.070874974,0.007557544,0.024956893,-0.012536875,-0.0092878705,0.008400664,-0.016711058,-0.0457814,0.037814904,0.0633747,-0.07401762,-0.022531627,0.00512724,-0.11766307,-0.018605428,-0.070915215,0.098230876,-0.050602224,-0.046869457,0.04856169,-0.038020477,0.014543746,0.04842452,-0.022996256,-0.009145798,-0.0005197498,-0.017328806,0.010275839,0.028327515,-0.04084515,0.04954195,-0.016362648,0.008979574,-0.030127324,-0.008421395,-0.05828275,0.10215089,0.061577268,0.052496668,0.032429215,0.0024723927,0.113524534,-0.067303225,-0.12981534,

In [24]:
# Now we'll see metadata filtering in action. 
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT id, text, metadata,
               embedding <=> %s::vector AS distance
        FROM chunks
        WHERE metadata->>'section' = 'Library'
        ORDER BY embedding <=> %s::vector
        LIMIT 2;
        """,
        (query_embedding.tolist(), query_embedding.tolist())
    )

    results = cur.fetchall()

for r in results:
    print(r)

('chunk_2', 'The library is open from 8 AM to 10 PM.', {'page': 10, 'section': 'Library'}, 0.39924633502960205)
('chunk_3', 'Students can borrow books from the university library.', {'page': 11, 'section': 'Library'}, 0.5542688039381016)


In [25]:
# Now we make a retrieval function that takes a query and returns the most similar chunks.
def retrieve(query, k=5):
    query_embedding = model.encode(query)

    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT
                id,
                text,
                metadata,
                embedding <=> %s::vector AS distance
            FROM chunks
            ORDER BY embedding <=> %s::vector
            LIMIT %s;
            """,
            (
                query_embedding.tolist(),
                query_embedding.tolist(),
                k
            )
        )

        return cur.fetchall()

In [26]:
# Testing the function
results = retrieve("When can I use the library?", k=2)

for result in results:
    print(result)

('chunk_2', 'The library is open from 8 AM to 10 PM.', {'page': 10, 'section': 'Library'}, 0.39924633502960205)
('chunk_3', 'Students can borrow books from the university library.', {'page': 11, 'section': 'Library'}, 0.5542688039381016)
